# 🏥 Chronic Kidney Disease Classifier Model Training (Google Colab)

This Jupyter Notebook was custom-developed for **Google Colab** to train and serialize the clinical predictive models used in the **Clini-SHAP Intelligent CDSS (Clinical Decision Support Suite)**.

### 📋 Dataset Context: Chronic Kidney Disease Database
### 🤖 Model Classifier Architecture: `Random Forest Classifier`

---

## ⚙️ Step 1: Environment Setup & Installations
We install the correct medical model dependencies including `xgboost`, `scikit-learn`, `shap`, `joblib`, `matplotlib`, and `seaborn` directly in Google Colab.


In [ ]:
# Install libraries in Google Colab
!pip install pandas numpy scikit-learn shap joblib matplotlib seaborn xgboost fpdf2


## 🧪 Step 2: Import Core Libraries
Load libraries required for numerical computation, exploratory data analysis (EDA), scaling pipeline, and SHAP interpretability.


In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
import shap
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

# Setup visualization parameters
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 📂 Step 3: Load Clinical Dataset
We fetch the dataset from the live repository and load it into a pandas DataFrame.

Loads the Chronic Kidney Disease Patient Dataset tracking red blood cells, specific gravity, pus cells, and urea indicators.


In [ ]:
# Load Chronic Kidney Disease Dataset
dataset_url = 'https://raw.githubusercontent.com/anushka06onu/AI-Healthcare-Analytics-Platform/main/data/kidney.csv'
try:
    df = pd.read_csv(dataset_url)
    print('Successfully downloaded kidney dataset!')
except Exception as e:
    print('Error loading:')
    raise e

print(f'Shape of Renal Patient Cohort: {df.shape}')
df.head()


## 🔬 Step 4: Train-Test Split and Clinical Preprocessing
We partition our EMR metrics into training and test cohorts (80% training, 20% test stratified) and scale features using `StandardScaler`.


In [ ]:
# Replace missing indicator '?' strings with standardized NumPy NaNs
df = df.replace('?', np.nan)

# Preprocess continuous numeric lab observations
num_cols = ['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc', 'sg', 'al', 'su']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

# Preprocess and map binary categorical lab markers
binary_cols = {
    'rbc': {'normal': 0, 'abnormal': 1},
    'pc': {'normal': 0, 'abnormal': 1},
    'pcc': {'notpresent': 0, 'present': 1},
    'ba': {'notpresent': 0, 'present': 1},
    'htn': {'no': 0, 'yes': 1},
    'dm': {'no': 0, 'yes': 1},
    'cad': {'no': 0, 'yes': 1},
    'pe': {'no': 0, 'yes': 1},
    'ane': {'no': 0, 'yes': 1},
    'appet': {'good': 0, 'poor': 1}
}

for col, mapping in binary_cols.items():
    df[col] = df[col].str.strip() if df[col].dtype == object else df[col]
    df[col] = df[col].map(mapping)
    mode_val = df[col].mode()[0] if not df[col].mode().empty else 0
    df[col] = df[col].fillna(mode_val).astype(int)

# Extract disease classification and clean labels
df['classification'] = df['classification'].str.strip()
df['target'] = df['classification'].map({'ckd': 1, 'notckd': 0})

# Drop mapping vectors
X = df.drop(columns=['classification', 'target'])
y = df['target']

# Perform stratified training split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale EMR physical records
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

print('Renal physical and urinalysis EMR pipeline successfully established!')


## 🤖 Step 5: Train EMR Predictive Classifier
We train the high-fidelity `Random Forest Classifier` model on our preprocessed training cohort.


In [ ]:
# Train kidney Random Forest classifier (per models/ diagnostic checks)
model = RandomForestClassifier(random_state=42, n_estimators=100)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

print('--- CLINICAL DIAGNOSTIC METRICS ---')
print(f'Accuracy Score:  {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision Score: {precision_score(y_test, y_pred):.4f}')
print(f'Recall Score:    {recall_score(y_test, y_pred):.4f}')
print(f'ROC-AUC Score:   {roc_auc_score(y_test, y_pred_proba):.4f}')


## 🔍 Step 6: Local and Global SHAP Attributions
To ensure 100% medical interpretability and accountabilities, we initialize SHAP explainer objects and pre-calculate SHAP matrices.


In [ ]:
# Random Forest models can use TreeExplainer directly (Fast and accurate!)
print('Running fast SHAP Tree Explainer attributions for Random Forest...')
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_scaled)

# Extract target index for probability of positive disease predictions
if isinstance(shap_values, list):
    shap_matrix = shap_values[1] if len(shap_values) > 1 else shap_values[0]
elif isinstance(shap_values, np.ndarray) and len(shap_values.shape) == 3:
    shap_matrix = shap_values[:, :, 1]
else:
    shap_matrix = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_matrix, X_test_scaled, show=False)
plt.title('Chronic Kidney Disease (CKD) Attributions (SHAP Beeswarm)', fontsize=14)
plt.tight_layout()
plt.show()


## 💾 Step 7: Export Serialized Components
We export the trained model classifier, the calibrated scaler, baseline training feature names, and pre-computed SHAP files for seamless integration into the CDSS application.


In [ ]:
os.makedirs('models', exist_ok=True)
os.makedirs('shap_files', exist_ok=True)

joblib.dump(model, 'models/kidney_model.pkl')
joblib.dump(scaler, 'models/kidney_scaler.pkl')
joblib.dump(list(scaler.feature_names_in_), 'models/kidney_columns.joblib')
joblib.dump(X_train, 'models/kidney_X_train.joblib')

joblib.dump(explainer, 'shap_files/kidney_explainer.joblib')
joblib.dump(shap_matrix, 'shap_files/kidney_shap_values.joblib')
joblib.dump(X_test_scaled, 'shap_files/kidney_X_test.joblib')

print('Renal EMR serializations successfully completed!')
